# 2) Interactive Adapter Training (Colab)

Widget-driven continual adapter training with live progress and OOD calibration.

In [ ]:
import importlib
import json
import os
import random
import shutil
import subprocess
import sys
import time
from pathlib import Path
from datetime import datetime

import matplotlib.pyplot as plt
import torch


from scripts.colab_repo_bootstrap import (
    install_colab_requirements,
    mount_drive_if_available,
    resolve_repo_root,
    running_in_colab,
)

def _ensure_int8_runtime_stack(in_colab: bool) -> tuple[bool, str, bool]:
    if not in_colab:
        return False, "not-colab", False

    repair_attempted = False

    try:
        import importlib.util as _ilu
        import transformers as _tf

        bnb_found = _ilu.find_spec("bitsandbytes") is not None
        tf_ver = str(getattr(_tf, "__version__", "0"))
        tf_major = int(tf_ver.split(".", 1)[0]) if tf_ver and tf_ver[0].isdigit() else 0
    except Exception:
        bnb_found = False
        tf_ver = "unknown"
        tf_major = 0

    needs_fix = (not bnb_found) or (tf_major >= 5)
    if needs_fix:
        repair_attempted = True
        print("[bootstrap] Repairing INT8 stack (requires transformers<5 and bitsandbytes)...")
        result = subprocess.run(
            [
                sys.executable,
                "-m",
                "pip",
                "install",
                "-q",
                "--upgrade",
                "transformers>=4.50.0,<5.0.0",
                "bitsandbytes>=0.43.0",
                "accelerate>=0.24.0",
                "peft>=0.6.0",
            ],
            check=False,
        )
        if result.returncode != 0:
            print("[bootstrap] WARNING: dependency repair command returned non-zero exit status.")
        importlib.invalidate_caches()

    try:
        import importlib.util as _ilu
        import transformers as _tf

        bnb_ready = _ilu.find_spec("bitsandbytes") is not None
        tf_ver_final = str(getattr(_tf, "__version__", "unknown"))
        tf_major_final = int(tf_ver_final.split(".", 1)[0]) if tf_ver_final and tf_ver_final[0].isdigit() else 0
        int8_ready = bnb_ready and (tf_major_final < 5)
        return int8_ready, tf_ver_final, repair_attempted
    except Exception:
        return False, "unknown", repair_attempted



def rt(msg: str) -> None:
    print(f"[{datetime.now().strftime('%H:%M:%S')}] {msg}")


IN_COLAB = running_in_colab()

rt("Cell 1: setup started")
if IN_COLAB:
    mount_drive_if_available(force_remount=False)

ROOT = resolve_repo_root()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

REQ = ROOT / "colab_notebooks" / "requirements_colab.txt"
install_colab_requirements(REQ, IN_COLAB)

INT8_BACKEND_READY, TRANSFORMERS_VERSION, RESTART_REQUIRED = _ensure_int8_runtime_stack(IN_COLAB)

if torch.cuda.is_available():
    DEVICE = "cuda"
    torch.backends.cudnn.benchmark = True
    torch.backends.cuda.matmul.allow_tf32 = True
else:
    DEVICE = "cpu"

rt(f"Repository root: {ROOT}")
rt(f"Device: {DEVICE}")
rt(f"transformers version: {TRANSFORMERS_VERSION}")
rt(f"bitsandbytes+INT8 stack ready: {INT8_BACKEND_READY}")

if RESTART_REQUIRED:
    raise RuntimeError(
        "Notebook dependencies were updated in this runtime. "
        "Please restart runtime, reopen notebook, then rerun from Cell 1."
    )

if DEVICE != "cuda":
    raise RuntimeError(
        "GPU runtime is required. This notebook refuses CPU execution to enforce training parity. "
        "In Colab: Runtime -> Change runtime type -> GPU (A100/T4/L4), then rerun Cell 1."
    )

rt("Cell 1: setup completed")

In [ ]:
import ipywidgets as widgets
from IPython.display import display

rt("Cell 2: building parameter panel")

BASE_CONFIG = json.loads((ROOT / "config" / "base.json").read_text(encoding="utf-8"))
TRAIN_CFG = BASE_CONFIG.get("training", {}).get("continual", {})
ADAPTER_CFG = TRAIN_CFG.get("adapter", {})
OOD_CFG = TRAIN_CFG.get("ood", {})

STATE = {
    "validated": False,
    "dataset_summary": None,
    "class_names": [],
    "runtime_dataset_root": None,
    "adapter": None,
    "loaders": None,
    "history": None,
    "calibration": None,
}

dataset_root_text = widgets.Text(value="data/class_root_dataset", description="Dataset:", layout=widgets.Layout(width="700px"))
crop_name_text = widgets.Text(value="tomato", description="Crop:")
epochs_slider = widgets.IntSlider(value=int(TRAIN_CFG.get("num_epochs", 1)), min=1, max=50, description="Epochs")
batch_size_slider = widgets.IntSlider(value=int(TRAIN_CFG.get("batch_size", 8)), min=1, max=64, description="Batch")
lr_slider = widgets.FloatLogSlider(value=float(TRAIN_CFG.get("learning_rate", 1e-4)), base=10, min=-6, max=-2, step=0.05, description="LR")
lora_r_slider = widgets.IntSlider(value=int(ADAPTER_CFG.get("lora_r", 16)), min=4, max=64, description="LoRA r")
lora_alpha_slider = widgets.IntSlider(value=int(ADAPTER_CFG.get("lora_alpha", 32)), min=8, max=128, description="LoRA α")
lora_dropout_slider = widgets.FloatSlider(value=float(ADAPTER_CFG.get("lora_dropout", 0.1)), min=0.0, max=0.5, step=0.01, description="Dropout")
ood_slider = widgets.FloatSlider(value=float(OOD_CFG.get("threshold_factor", 2.0)), min=1.0, max=5.0, step=0.1, description="OOD x")
allow_cpu_fallback_checkbox = widgets.Checkbox(value=not bool(globals().get("INT8_BACKEND_READY", False)), description="Allow non-INT8 fallback", indent=False)
validate_btn = widgets.Button(description="Validate & Start Training", button_style="success")
status_out = widgets.Output()
guide_html = widgets.HTML(value=(
    "<b>Parameter Guide (Suggested Values)</b><br>"
    "<ul style='margin-top:6px'>"
    "<li><b>Dataset</b>: class-root folder like <code>data/class_root_dataset</code> (required).</li>"
    "<li><b>Crop</b>: crop adapter name (suggested: <code>tomato</code>, <code>pepper</code>, <code>corn</code>).</li>"
    "<li><b>Epochs</b>: total training passes (start with <b>3-10</b>, increase if underfitting).</li>"
    "<li><b>Batch</b>: images per step (suggested <b>8</b> on T4/16GB, <b>16-32</b> on larger GPUs).</li>"
    "<li><b>LR</b>: learning rate (safe default <b>1e-4</b>; try <b>5e-5</b> if unstable).</li>"
    "<li><b>LoRA r</b>: adapter rank (suggested <b>16</b>; try <b>8</b> for speed, <b>32</b> for capacity).</li>"
    "<li><b>LoRA α</b>: scaling strength (suggested <b>32</b>; often about 2x <code>r</code>).</li>"
    "<li><b>Dropout</b>: LoRA regularization (suggested <b>0.1</b>; use <b>0.05-0.2</b>).</li>"
    "<li><b>OOD x</b>: OOD threshold factor (default <b>2.0</b>; increase to reduce false OOD).</li>"
    "<li><b>Allow non-INT8 fallback</b>: enable only when bitsandbytes backend is unavailable in Colab; keep off for strict production parity.</li>"
    "</ul>"
    "Click <i>Validate & Start Training</i>, then run Cell 3 for dataset checks."
))


def gather_widget_config():
    return {
        "dataset_root": dataset_root_text.value.strip(),
        "crop_name": crop_name_text.value.strip(),
        "num_epochs": int(epochs_slider.value),
        "batch_size": int(batch_size_slider.value),
        "learning_rate": float(lr_slider.value),
        "lora_r": int(lora_r_slider.value),
        "lora_alpha": int(lora_alpha_slider.value),
        "lora_dropout": float(lora_dropout_slider.value),
        "ood_threshold_factor": float(ood_slider.value),
        "allow_cpu_fallback": bool(allow_cpu_fallback_checkbox.value),
    }


def _on_validate_clicked(_):
    cfg = gather_widget_config()
    STATE["widget_config"] = cfg
    with status_out:
        status_out.clear_output()
        print("Config captured. Run Cell 3 for full dataset validation details.")
        print(f"Timestamp: {datetime.now().strftime('%H:%M:%S')}")


validate_btn.on_click(_on_validate_clicked)

panel = widgets.VBox([
    guide_html,
    dataset_root_text,
    crop_name_text,
    widgets.HBox([epochs_slider, batch_size_slider]),
    widgets.HBox([lr_slider, lora_r_slider, lora_alpha_slider]),
    widgets.HBox([lora_dropout_slider, ood_slider]),
    allow_cpu_fallback_checkbox,
    validate_btn,
    status_out,
])

display(panel)
rt("Cell 2: parameter panel ready")

In [ ]:
from scripts.evaluate_dataset_layout import evaluate_layout

rt("Cell 3: dataset validation started")

cfg = STATE.get("widget_config") or {
    "dataset_root": dataset_root_text.value.strip(),
    "crop_name": crop_name_text.value.strip(),
}

raw_root = Path(cfg["dataset_root"]).expanduser()
if not raw_root.is_absolute():
    raw_root = (ROOT / raw_root).resolve()

summary = evaluate_layout(raw_root)
STATE["dataset_summary"] = summary

print("Dataset root:", raw_root)
print("Layout ok:", summary.get("ok"))
print("Summary:", summary.get("summary", {}))

if summary.get("errors"):
    print("Errors:")
    for item in summary["errors"]:
        print(" -", item)

if summary.get("warnings"):
    print("Warnings:")
    for item in summary["warnings"]:
        print(" -", item)

class_names = [entry.get("class_name") for entry in summary.get("classes", []) if entry.get("class_name")]
STATE["class_names"] = class_names
print("Discovered classes:", class_names)

STATE["validated"] = bool(summary.get("ok")) and len(class_names) > 0
rt(f"Cell 3: validation completed (ok={STATE['validated']})")

In [ ]:
from src.adapter.independent_crop_adapter import IndependentCropAdapter
from src.utils.data_loader import create_training_loaders

rt("Cell 4: engine initialization started")


def prepare_runtime_dataset_layout(
    class_root: Path,
    crop_name: str,
    seed: int = 42,
    force_rebuild: bool = False,
    allowed_class_names: list[str] | None = None,
) -> Path:
    runtime_root = ROOT / "data" / "runtime_notebook_datasets"
    crop_root = runtime_root / crop_name
    metadata_path = crop_root / "_split_metadata.json"

    class_dirs = sorted([p for p in class_root.iterdir() if p.is_dir()])
    allowed_set = set(allowed_class_names or [])
    source_key = {
        "layout_version": 3,
        "class_root": str(class_root.resolve()),
        "crop_name": crop_name,
        "seed": int(seed),
        "class_dirs": [p.name for p in class_dirs],
        "allowed_class_names": sorted(list(allowed_set)),
        "split_policy": "continual=80,val=10,test=10",
    }

    if (not force_rebuild) and crop_root.exists() and metadata_path.exists():
        try:
            cached = json.loads(metadata_path.read_text(encoding="utf-8"))
        except Exception:
            cached = {}
        if cached == source_key:
            rt("Reusing cached runtime dataset split layout")
            return runtime_root

    if crop_root.exists():
        shutil.rmtree(crop_root)
    crop_root.mkdir(parents=True, exist_ok=True)

    rng = random.Random(seed)
    split_names = ("continual", "val", "test")
    image_exts = {".jpg", ".jpeg", ".png", ".bmp", ".webp", ".tif", ".tiff"}

    def materialize_file(src_path: Path, dst: Path) -> None:
        try:
            os.symlink(src_path, dst)
            return
        except OSError:
            pass

        try:
            os.link(src_path, dst)
            return
        except OSError:
            pass

        shutil.copy2(src_path, dst)

    used_target_class_names = set()
    for cls_dir in class_dirs:
        normalized_class_name = normalize_class_name(cls_dir.name)
        if not normalized_class_name:
            continue
        if allowed_set and normalized_class_name not in allowed_set:
            continue
        if normalized_class_name in used_target_class_names:
            rt(f"Skipping duplicate normalized class name: {cls_dir.name} -> {normalized_class_name}")
            continue

        rt(f"Preparing dataset split for class: {cls_dir.name}")
        image_files = [
            p for p in cls_dir.rglob("*")
            if p.is_file() and p.suffix.lower() in image_exts
        ]
        rng.shuffle(image_files)
        n = len(image_files)
        if n == 0:
            continue

        train_n = max(1, int(0.8 * n))
        val_n = max(1, int(0.1 * n)) if n >= 3 else 0
        if train_n + val_n >= n:
            val_n = max(0, n - train_n - 1)
        test_n = max(0, n - train_n - val_n)

        split_map = {
            "continual": image_files[:train_n],
            "val": image_files[train_n:train_n + val_n],
            "test": image_files[train_n + val_n:train_n + val_n + test_n],
        }

        for split in split_names:
            target = crop_root / split / normalized_class_name
            target.mkdir(parents=True, exist_ok=True)
            for src_path in split_map[split]:
                dst = target / src_path.name
                if dst.exists():
                    continue
                materialize_file(src_path, dst)
        used_target_class_names.add(normalized_class_name)

    metadata_path.write_text(json.dumps(source_key, indent=2), encoding="utf-8")
    return runtime_root


if not STATE.get("validated"):
    raise RuntimeError("Run Cell 3 and ensure dataset validation passes first.")

cfg = STATE["widget_config"]
crop_name = cfg["crop_name"].strip().lower()
class_root = Path(cfg["dataset_root"]).expanduser()
if not class_root.is_absolute():
    class_root = (ROOT / class_root).resolve()

def normalize_class_name(name: str) -> str:
    normalized = "".join(ch.lower() if ch.isalnum() else "_" for ch in str(name).strip())
    while "__" in normalized:
        normalized = normalized.replace("__", "_")
    return normalized.strip("_")

available_class_names = sorted({
    normalize_class_name(p.name)
    for p in class_root.iterdir()
    if p.is_dir() and normalize_class_name(p.name)
})
taxonomy_path = ROOT / "config" / "plant_taxonomy.json"
expected_for_crop = None
try:
    taxonomy = json.loads(taxonomy_path.read_text(encoding="utf-8"))
    disease_map = taxonomy.get("crop_specific_diseases", {})
    crop_diseases = disease_map.get(crop_name, []) if isinstance(disease_map, dict) else []
    if isinstance(crop_diseases, list) and crop_diseases:
        normalized_diseases = [normalize_class_name(name) for name in crop_diseases]
        expected_for_crop = sorted({"healthy", *[name for name in normalized_diseases if name]})
except Exception as exc:
    rt(f"Taxonomy lookup failed ({exc}); using available class folders.")

if expected_for_crop:
    aligned_class_names = [name for name in expected_for_crop if name in available_class_names]
else:
    aligned_class_names = available_class_names

if not aligned_class_names:
    raise RuntimeError(
        f"No usable classes found for crop '{crop_name}'. Available normalized classes: {available_class_names}"
    )

STATE["class_names"] = aligned_class_names
print("Training classes:", aligned_class_names)

runtime_data_root = prepare_runtime_dataset_layout(
    class_root=class_root,
    crop_name=crop_name,
    force_rebuild=os.environ.get("AADS_FORCE_REBUILD_SPLITS", "0") == "1",
    allowed_class_names=aligned_class_names,
)
STATE["runtime_dataset_root"] = runtime_data_root
rt("Runtime dataset layout prepared")

continual_cfg = json.loads(json.dumps(BASE_CONFIG.get("training", {}).get("continual", {})))
continual_cfg["backbone"]["model_name"] = continual_cfg.get("backbone", {}).get("model_name", "facebook/dinov3-vitl16-pretrain-lvd1689m")
continual_cfg["device"] = DEVICE
continual_cfg["num_epochs"] = int(cfg["num_epochs"])
continual_cfg["batch_size"] = int(cfg["batch_size"])
continual_cfg["learning_rate"] = float(cfg["learning_rate"])
continual_cfg["adapter"]["lora_r"] = int(cfg["lora_r"])
continual_cfg["adapter"]["lora_alpha"] = int(cfg["lora_alpha"])
continual_cfg["adapter"]["lora_dropout"] = float(cfg["lora_dropout"])
continual_cfg["ood"]["threshold_factor"] = float(cfg["ood_threshold_factor"])
continual_cfg.setdefault("quantization", {})["strict_backend"] = not bool(cfg.get("allow_cpu_fallback", False))
continual_cfg.setdefault("quantization", {})["allow_cpu_fallback"] = bool(cfg.get("allow_cpu_fallback", False))

adapter = IndependentCropAdapter(crop_name=crop_name, device=DEVICE)
rt("Initializing adapter engine (this may download/load backbone)")
adapter.initialize_engine(class_names=STATE["class_names"], config={"training": {"continual": continual_cfg}})
rt("Adapter engine initialized")

loaders = create_training_loaders(
    data_dir=str(runtime_data_root),
    crop=crop_name,
    batch_size=int(cfg["batch_size"]),
    num_workers=0,
    use_cache=False,
)
rt("Training/validation/test loaders created")

STATE["adapter"] = adapter
STATE["loaders"] = loaders
STATE["continual_config"] = continual_cfg

print("Engine initialized.")
print("Runtime dataset root:", runtime_data_root)
print("Classes:", len(STATE["class_names"]))
print("Resolved LoRA target modules:", len(adapter.target_modules_resolved))
print("Fusion layers:", continual_cfg.get("fusion", {}).get("layers"))
print("Fusion output dim:", continual_cfg.get("fusion", {}).get("output_dim"))

trainer = adapter._trainer
if trainer is not None:
    total_params = 0
    trainable_params = 0
    for p in trainer.adapter_model.parameters():
        total_params += p.numel()
        if p.requires_grad:
            trainable_params += p.numel()
    for p in trainer.classifier.parameters():
        total_params += p.numel()
        if p.requires_grad:
            trainable_params += p.numel()
    for p in trainer.fusion.parameters():
        total_params += p.numel()
        if p.requires_grad:
            trainable_params += p.numel()

    print(f"Total params (adapter+heads): {total_params:,}")
    print(f"Trainable params: {trainable_params:,}")

rt("Cell 4: engine initialization completed")

In [ ]:
from IPython.display import clear_output, display

rt("Cell 5: training started")

if STATE.get("adapter") is None or STATE.get("loaders") is None:
    raise RuntimeError("Run Cell 4 first.")

adapter = STATE["adapter"]
loaders = STATE["loaders"]
num_epochs = int(STATE["widget_config"]["num_epochs"])

progress_bar = widgets.IntProgress(value=0, min=0, max=max(1, len(loaders["train"])), description="Batch")
status_html = widgets.HTML(value="<b>Ready.</b>")
loss_out = widgets.Output()
display(progress_bar, status_html, loss_out)

train_loss_curve = []
start_time = time.time()


def progress_cb(event):
    if "batch" in event:
        progress_bar.max = max(1, int(event.get("total_batches", 1)))
        progress_bar.value = int(event.get("batch", 0))

        elapsed = time.time() - start_time
        batches_done = max(1, ((event.get("epoch", 1) - 1) * progress_bar.max) + event.get("batch", 1))
        total_batches = max(1, num_epochs * progress_bar.max)
        eta = (elapsed / batches_done) * max(0, total_batches - batches_done)

        status_html.value = (
            f"<b>Epoch:</b> {event.get('epoch', 0)} | "
            f"<b>Batch:</b> {event.get('batch', 0)}/{event.get('total_batches', 0)} | "
            f"<b>Batch Loss:</b> {event.get('batch_loss', 0.0):.4f} | "
            f"<b>Elapsed:</b> {elapsed:.1f}s | <b>ETA:</b> {eta:.1f}s"
        )

    if "epoch_done" in event:
        train_loss_curve.append(float(event["epoch_loss"]))
        with loss_out:
            clear_output(wait=True)
            plt.figure(figsize=(6, 3))
            plt.plot(range(1, len(train_loss_curve) + 1), train_loss_curve, marker="o")
            plt.xlabel("Epoch")
            plt.ylabel("Loss")
            plt.title("Training Loss")
            plt.grid(True, alpha=0.3)
            plt.show()


train_result = adapter.train_increment(
    loaders["train"],
    num_epochs=num_epochs,
    progress_callback=progress_cb,
)

elapsed_total = time.time() - start_time
STATE["history"] = train_result.get("history", {})

print("Training complete")
print("Total time (s):", round(elapsed_total, 2))
print("History:", STATE["history"])
rt("Cell 5: training completed")

In [ ]:
from IPython.display import display

rt("Cell 6: OOD calibration started")

if STATE.get("adapter") is None or STATE.get("loaders") is None:
    raise RuntimeError("Run Cell 4 first.")

adapter = STATE["adapter"]
val_loader = STATE["loaders"].get("val")

if val_loader is None or len(val_loader.dataset) == 0:
    raise RuntimeError("Validation loader is empty; cannot calibrate OOD.")

cal = adapter.calibrate_ood(val_loader)
STATE["calibration"] = cal

num_classes = cal.get("ood_calibration", {}).get("num_classes", 0)
version = cal.get("ood_calibration", {}).get("version", 0)
status_color = "green" if int(num_classes) > 0 else "red"

cal_html = widgets.HTML(
    value=f"<b style='color:{status_color}'>OOD calibration completed</b> | classes={num_classes} | version={version}"
)
display(cal_html)
rt("Cell 6: OOD calibration completed")

In [ ]:
rt("Cell 7: adapter save started")

if STATE.get("adapter") is None:
    raise RuntimeError("Run Cell 4 first.")

checkpoint_dir = ROOT / "outputs" / "colab_notebook_training"
checkpoint_dir.mkdir(parents=True, exist_ok=True)

STATE["adapter"].save_adapter(str(checkpoint_dir))
asset_dir = checkpoint_dir / "continual_sd_lora_adapter"

print("Saved adapter directory:", asset_dir)
if not asset_dir.exists():
    raise RuntimeError(f"Expected adapter dir missing: {asset_dir}")

for p in sorted(asset_dir.rglob("*")):
    if p.is_file():
        print(" -", p.relative_to(ROOT))

rt("Cell 7: adapter save completed")

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix

rt("Cell 8: validation metrics started")

if STATE.get("adapter") is None or STATE.get("loaders") is None:
    raise RuntimeError("Run Cell 4 first.")

adapter = STATE["adapter"]
val_loader = STATE["loaders"].get("val")
if val_loader is None or len(val_loader.dataset) == 0:
    raise RuntimeError("Validation loader is empty.")

trainer = adapter._trainer
if trainer is None:
    raise RuntimeError("Trainer not initialized.")

trainer.adapter_model.eval()
trainer.classifier.eval()
trainer.fusion.eval()

y_true = []
y_pred = []

with torch.no_grad():
    for batch in val_loader:
        images = batch["images"].to(trainer.device)
        labels = batch["labels"].to(trainer.device)
        logits = trainer.forward_logits(images)
        preds = torch.argmax(logits, dim=1)
        y_true.extend(labels.cpu().tolist())
        y_pred.extend(preds.cpu().tolist())

if not y_true:
    raise RuntimeError("No validation samples available.")

classes = [name for name, _ in sorted(adapter.class_to_idx.items(), key=lambda kv: kv[1])]
print(classification_report(y_true, y_pred, target_names=classes, zero_division=0))

cm = confusion_matrix(y_true, y_pred)
plt.figure(figsize=(6, 5))
plt.imshow(cm, cmap="Blues")
plt.title("Validation Confusion Matrix")
plt.xlabel("Predicted")
plt.ylabel("True")
plt.xticks(range(len(classes)), classes, rotation=45, ha="right")
plt.yticks(range(len(classes)), classes)
for i in range(cm.shape[0]):
    for j in range(cm.shape[1]):
        plt.text(j, i, int(cm[i, j]), ha="center", va="center")
plt.tight_layout()
plt.show()
rt("Cell 8: validation metrics completed")